# Cross-Modal Retrieval: Teach CLIP a New Visual Domain

**Goal:** Show that a smaller, fine-tuned CLIP model can match or beat a larger CLIP model on domain-specific image retrieval.

**Setup:**
- **Large model:** `openai/clip-vit-large-patch14` (428M params)
- **Small model:** `openai/clip-vit-base-patch32` (151M params, ~3x smaller)
- **Dataset:** RSICD — satellite/aerial imagery with text captions. CLIP wasn't trained on this domain.

**Plan:**
1. Evaluate both CLIP models as baselines — both should struggle on satellite images
2. Fine-tune the smaller CLIP with random, hard, and mixed negatives
3. Compare: can fine-tuned small CLIP beat large CLIP on this domain?

In [ ]:
!pip install khoji Pillow

## Step 1: Baseline Evaluation

Neither CLIP model was trained on satellite imagery. Let's see how they perform out-of-the-box.

In [ ]:
from khoji import MultimodalEvaluator
from khoji.multimodal_dataset import load_rsicd

# Load RSICD test set
test_dataset = load_rsicd(split="test")
print(f"Test: {len(test_dataset.queries)} queries, {len(test_dataset.corpus)} images")

In [ ]:
# Baseline 1: CLIP ViT-L/14 (428M params) — the "big" model
large_evaluator = MultimodalEvaluator("openai/clip-vit-large-patch14")
large_baseline = large_evaluator.evaluate(
    dataset_name="rsicd",
    k_values=[1, 5, 10],
    dataset=test_dataset,
)
large_baseline.print()
del large_evaluator

In [ ]:
# Baseline 2: CLIP ViT-B/32 (151M params, ~3x smaller) — the model we'll fine-tune
small_evaluator = MultimodalEvaluator("openai/clip-vit-base-patch32")
small_baseline = small_evaluator.evaluate(
    dataset_name="rsicd",
    k_values=[1, 5, 10],
    dataset=test_dataset,
)
small_baseline.print()
del small_evaluator

In [ ]:
# Compare baselines
print("Baseline comparison (no fine-tuning):")
print(f"{'Metric':<12} {'CLIP-L (428M)':>14} {'CLIP-B (151M)':>15} {'Gap':>10}")
print("-" * 53)
for metric in large_baseline.metrics:
    l = large_baseline.metrics[metric]
    s = small_baseline.metrics[metric]
    print(f"{metric:<12} {l:>14.4f} {s:>15.4f} {s - l:>+10.4f}")

## Step 2: Fine-tune Small CLIP — Three Strategies

### Approach 1: Random Negatives via YAML Config

In [ ]:
config_yaml = """
model:
  name: openai/clip-vit-base-patch32
  lora_target: both

data:
  dataset: arampacha/rsicd
  split: train
  negatives: random
  n_negatives: 3

lora:
  r: 16
  alpha: 32
  dropout: 0.1

train:
  epochs: 3
  batch_size: 16
  grad_accum_steps: 2
  lr: 2e-5
  warmup_steps: 50
  max_length: 77
  loss: infonce
  temperature: 0.05
  sanity_check_samples: 5

seed: 42

eval:
  k_values: [1, 5, 10]
  split: test
  run_before: false
  run_after: true

output_dir: ./output/rsicd-random
"""

with open("rsicd_random.yaml", "w") as f:
    f.write(config_yaml)

print("Config written.")

In [ ]:
from khoji import MultimodalForgeConfig
from khoji.multimodal_run import run_multimodal

config = MultimodalForgeConfig.from_yaml("rsicd_random.yaml")
result_random = run_multimodal(config)

### Approach 2: Hard Negatives via Python API

In [ ]:
from functools import partial
from khoji import (
    MultimodalEmbeddingModel,
    MultimodalTrainer,
    MultimodalTrainingConfig,
    MultimodalTripletDataset,
    LoRASettings,
)
from khoji.multimodal_data import mine_hard_negatives_multimodal
from khoji.multimodal_dataset import load_rsicd
from khoji.loss import infonce_loss

# Load training data
train_dataset = load_rsicd(split="train")
print(f"Train: {len(train_dataset.queries)} queries, {len(train_dataset.corpus)} images")

# Mine hard negatives
mining_model = MultimodalEmbeddingModel("openai/clip-vit-base-patch32")
hard_triplets = mine_hard_negatives_multimodal(
    train_dataset, mining_model,
    n_negatives=3, top_k=50,
)
del mining_model
print(f"Mined {len(hard_triplets)} hard negative triplets")

In [ ]:
# Train
config_hard = MultimodalTrainingConfig(
    epochs=3,
    batch_size=16,
    grad_accum_steps=2,
    lr=2e-5,
    warmup_steps=50,
    max_length=77,
    loss_fn=partial(infonce_loss, temperature=0.05),
    lora=LoRASettings(r=16, alpha=32, dropout=0.1),
    lora_target="both",
    save_dir="./output/rsicd-hard/adapter",
    sanity_check_samples=5,
    base_dir=train_dataset.base_dir,
)

trainer_hard = MultimodalTrainer("openai/clip-vit-base-patch32", config_hard)
history_hard = trainer_hard.train(MultimodalTripletDataset(hard_triplets))

In [ ]:
# Evaluate
eval_hard = MultimodalEvaluator(
    "openai/clip-vit-base-patch32",
    adapter_path="./output/rsicd-hard/adapter",
)
result_hard_eval = eval_hard.evaluate(
    dataset_name="rsicd", k_values=[1, 5, 10], dataset=test_dataset,
)
result_hard_eval.print()
del eval_hard

### Approach 3: Mixed Negatives via Python API

In [ ]:
from khoji.multimodal_data import build_mixed_negatives_multimodal

# Build mixed negatives: 2 random + 1 hard
mining_model = MultimodalEmbeddingModel("openai/clip-vit-base-patch32")
mixed_triplets = build_mixed_negatives_multimodal(
    train_dataset, mining_model,
    n_random=2, n_hard=1, top_k=50,
)
del mining_model
print(f"Built {len(mixed_triplets)} mixed triplets")

In [ ]:
# Train
config_mixed = MultimodalTrainingConfig(
    epochs=3,
    batch_size=16,
    grad_accum_steps=2,
    lr=2e-5,
    warmup_steps=50,
    max_length=77,
    loss_fn=partial(infonce_loss, temperature=0.05),
    lora=LoRASettings(r=16, alpha=32, dropout=0.1),
    lora_target="both",
    save_dir="./output/rsicd-mixed/adapter",
    sanity_check_samples=5,
    base_dir=train_dataset.base_dir,
)

trainer_mixed = MultimodalTrainer("openai/clip-vit-base-patch32", config_mixed)
history_mixed = trainer_mixed.train(MultimodalTripletDataset(mixed_triplets))

In [ ]:
# Evaluate
eval_mixed = MultimodalEvaluator(
    "openai/clip-vit-base-patch32",
    adapter_path="./output/rsicd-mixed/adapter",
)
result_mixed_eval = eval_mixed.evaluate(
    dataset_name="rsicd", k_values=[1, 5, 10], dataset=test_dataset,
)
result_mixed_eval.print()
del eval_mixed

## Step 3: Compare All Results

In [ ]:
results = {
    "CLIP-L (428M)": large_baseline.metrics,
    "CLIP-B baseline": small_baseline.metrics,
    "CLIP-B + random": result_random.finetuned.metrics,
    "CLIP-B + hard": result_hard_eval.metrics,
    "CLIP-B + mixed": result_mixed_eval.metrics,
}

header = f"{'Metric':<12}" + "".join(f"{name:>18}" for name in results)
print(header)
print("-" * len(header))

for metric in ["ndcg@10", "mrr@10", "recall@10"]:
    row = f"{metric:<12}"
    for name, metrics in results.items():
        row += f"{metrics[metric]:>18.4f}"
    print(row)

# Gap analysis
large_ndcg = large_baseline.metrics["ndcg@10"]
small_ndcg = small_baseline.metrics["ndcg@10"]
best_ft_ndcg = max(
    result_random.finetuned.metrics["ndcg@10"],
    result_hard_eval.metrics["ndcg@10"],
    result_mixed_eval.metrics["ndcg@10"],
)
gap_before = large_ndcg - small_ndcg
gap_after = large_ndcg - best_ft_ndcg
print(f"\nnDCG@10 gap vs CLIP-Large: {gap_before:.4f} (before) → {gap_after:.4f} (after)")
if gap_before > 0:
    closed = 100 * (1 - gap_after / gap_before)
    print(f"Gap closed by {closed:.1f}%")
    if gap_after < 0:
        print("Fine-tuned small CLIP SURPASSED the large CLIP!")

## Step 4: Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

histories = {
    "Random": result_random.history,
    "Hard": history_hard,
    "Mixed": history_mixed,
}

for name, hist in histories.items():
    axes[0].plot(hist.step_loss, label=name, alpha=0.7)
    axes[1].plot(hist.step_lr, label=name, alpha=0.7)
    axes[2].plot(hist.step_grad_norm, label=name, alpha=0.7)

axes[0].set_title("Loss per Step")
axes[1].set_title("Learning Rate")
axes[2].set_title("Gradient Norm")
for ax in axes:
    ax.set_xlabel("Optimizer Step")
    ax.legend()

plt.tight_layout()
plt.savefig("rsicd_training_curves.png", dpi=150)
plt.show()

## Step 5: Inference — Query Satellite Images with Text

In [ ]:
import torch

# Load fine-tuned model
model = MultimodalEmbeddingModel(
    "openai/clip-vit-base-patch32",
    adapter_path="./output/rsicd-mixed/adapter",
)

# Query: find satellite images matching a text description
queries = [
    "an airport with multiple runways",
    "a river flowing through a dense forest",
    "residential buildings in a suburban area",
]

# Encode queries
query_embs = model.encode_text(queries)

# Encode test corpus images
corpus_ids = list(test_dataset.corpus.keys())
corpus_sources = list(test_dataset.corpus.values())
image_embs = model.encode_image_sources(
    corpus_sources[:200],  # subset for speed
    base_dir=test_dataset.base_dir,
)

# Rank images for each query
scores = torch.mm(query_embs, image_embs.t())

for i, query in enumerate(queries):
    top5 = torch.topk(scores[i], k=5).indices.tolist()
    print(f"\nQuery: '{query}'")
    for rank, idx in enumerate(top5):
        print(f"  Rank {rank+1}: {corpus_sources[idx]} (score={scores[i][idx]:.4f})")

## Summary

| Model | Params | Fine-tuned | Domain | nDCG@10 |
|-------|--------|------------|--------|--------|
| CLIP ViT-L/14 | 428M | No | General | (baseline) |
| CLIP ViT-B/32 | 151M | No | General | (lower) |
| CLIP ViT-B/32 | 151M | Random negs | Satellite | (improved) |
| CLIP ViT-B/32 | 151M | Hard negs | Satellite | (improved) |
| CLIP ViT-B/32 | 151M | Mixed negs | Satellite | (best) |

**Key takeaway:** CLIP models struggle on domain-specific imagery they weren't trained on. A smaller CLIP, fine-tuned with LoRA on just a few thousand domain images, can match or surpass a 3x larger model. The adapter adds only a few MB to the 151M parameter model.